# Clean Length Measurement Analysis - Version 2

**Data channels:**
- Channel 1: Length measurement (voltage)
- Channel 2: Force signal (voltage)
- Control voltage: Expected square wave (+0.6V → -0.8V, 300s each)

**Linear relationship:** `length (mm) = -0.04130 × voltage`

**Analysis:** 3 focused figures (no drift correction - no slippage detected)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from sklearn.linear_model import LinearRegression
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = [14, 8]

In [ ]:
# Load and prepare data
df = pd.read_csv('20260430_fixedforce_length_measurement.csv')
print(f"Loaded {len(df)} data points over {df['elapsed_s'].max()/60:.1f} minutes")

# Apply median filtering to reduce noise
filter_size = 15
df['length_filtered'] = median_filter(df['ch1_MEAN_V'], size=filter_size)
df['force_filtered'] = median_filter(df['ch2_MEAN_V'], size=filter_size)

# Conversion factor: length (mm) = -0.04130 * voltage
length_conversion = -0.04130
df['length_mm'] = length_conversion * df['length_filtered']

print(f"Length range: {df['length_mm'].min():.2f} to {df['length_mm'].max():.2f} mm")
print(f"Voltage range: {df['ch1_MEAN_V'].min():.3f} to {df['ch1_MEAN_V'].max():.3f} V")

In [ ]:
# Detect cycle start (when square wave begins)
# Look for significant changes in the expected pattern
force_diff = np.abs(np.diff(df['force_filtered']))
change_threshold = np.percentile(force_diff, 95)  # Top 5% of changes
significant_changes = np.where(force_diff > change_threshold)[0]

if len(significant_changes) > 0:
    cycle_start_idx = significant_changes[0]
    cycle_start_time = df['elapsed_s'].iloc[cycle_start_idx]
    print(f"Detected cycle start at: {cycle_start_time:.1f} seconds")
else:
    cycle_start_time = 30  # Fallback estimate
    print(f"Using estimated cycle start: {cycle_start_time:.1f} seconds")

# Create control voltage pattern (expected square wave)
# VERSION 2: Starts with +0.6V, then -0.8V
control_voltage = np.zeros_like(df['elapsed_s'])
cycle_mask = df['elapsed_s'] >= cycle_start_time

for i, t in enumerate(df['elapsed_s']):
    if t >= cycle_start_time:
        time_since_start = t - cycle_start_time
        cycle_position = time_since_start % 600  # 600s = one complete cycle
        
        if cycle_position < 300:
            control_voltage[i] = 0.6   # First 300s: +0.6V
        else:
            control_voltage[i] = -0.8  # Next 300s: -0.8V
    else:
        control_voltage[i] = 0.0  # Before cycles start

df['control_voltage'] = control_voltage
print(f"Created control voltage pattern: {np.unique(control_voltage)}V")
print(f"Pattern: +0.6V (first 300s) → -0.8V (next 300s) → repeat")

In [ ]:
# FIGURE 1: Length voltage over time with dual axes and control voltage overlay
fig, ax1 = plt.subplots(figsize=(16, 8))

# Primary plot: Length voltage (filtered)
color = 'tab:blue'
ax1.set_xlabel('Time (seconds)', fontsize=14)
ax1.set_ylabel('Length Voltage (V)', color=color, fontsize=14)
line1 = ax1.plot(df['elapsed_s'], df['length_filtered'], color=color, linewidth=2, 
                 alpha=0.8, label='Length (Filtered)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Secondary Y-axis: Length in mm
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Length (mm)', color=color, fontsize=14)
ax2.plot(df['elapsed_s'], df['length_mm'], color=color, linewidth=2, alpha=0.8, label='Length (mm)')
ax2.tick_params(axis='y', labelcolor=color)

# Overlay control voltage on top
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 60))
color = 'tab:red'
ax3.set_ylabel('Control Voltage (V)', color=color, fontsize=14)
ax3.plot(df['elapsed_s'], df['control_voltage'], color=color, linewidth=3, 
         alpha=0.7, label='Control Square Wave', linestyle='--')
ax3.tick_params(axis='y', labelcolor=color)
ax3.set_ylim(-1.2, 1.0)

# Mark cycle start
ax1.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)
ax1.text(cycle_start_time + 10, ax1.get_ylim()[1]*0.9, f'Cycle Start\n({cycle_start_time:.0f}s)', 
         fontsize=12, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax1.set_title('Length Measurement with Control Voltage Pattern (+0.6V first)', fontsize=16, fontweight='bold', pad=20)

# Create combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax3.get_legend_handles_labels()
ax1.legend(lines1 + lines2 + lines3, labels1 + labels2 + labels3, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 2: Linear relationship validation (no drift correction needed)
# Focus on data after the initial offset for clean analysis
cycle_data = df[df['elapsed_s'] >= cycle_start_time].copy()

# Expected linear relationship
expected_slope = -0.04130

# Use filtered data for analysis
X = cycle_data['control_voltage'].values.reshape(-1, 1)
y = cycle_data['length_filtered'].values

# Fit linear model
model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
from sklearn.metrics import r2_score
r2 = r2_score(y, y_pred)

# Extract coefficients
measured_slope = model.coef_[0]
measured_intercept = model.intercept_

# Calculate statistics
from scipy import stats
correlation, p_value = stats.pearsonr(cycle_data['control_voltage'], cycle_data['length_filtered'])
rmse = np.sqrt(np.mean((y - y_pred)**2))
mae = np.mean(np.abs(y - y_pred))

fig, ax = plt.subplots(figsize=(16, 8))

# Scatter plot with time coloring
scatter = ax.scatter(cycle_data['control_voltage'], cycle_data['length_filtered'], 
                    alpha=0.6, s=30, c=cycle_data['elapsed_s'], cmap='plasma')

# Plot both expected and measured lines
voltage_range = np.linspace(cycle_data['control_voltage'].min(), 
                           cycle_data['control_voltage'].max(), 100)
expected_line = expected_slope * voltage_range
measured_line = measured_slope * voltage_range + measured_intercept

ax.plot(voltage_range, expected_line, 'r--', linewidth=4, 
        label=f'Expected: length = {expected_slope:.5f} × voltage', alpha=0.9)
ax.plot(voltage_range, measured_line, 'g-', linewidth=4, 
        label=f'Measured: length = {measured_slope:.5f} × voltage + {measured_intercept:.3f}', alpha=0.9)

ax.set_xlabel('Control Voltage (V)', fontsize=14)
ax.set_ylabel('Length Voltage (V)', fontsize=14)
ax.set_title(f'Linear Relationship Validation (R² = {r2:.4f}) - No Drift Correction', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Time (seconds)', fontsize=12)

# Add statistics text box
slope_error = abs(measured_slope - expected_slope) / abs(expected_slope) * 100
textstr = f'Slope Error: {slope_error:.1f}%\nR² = {r2:.6f}\nRMSE = {rmse:.6f}V\nCorrelation = {correlation:.6f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("Linear Relationship Analysis:")
print("=" * 50)
print(f"Expected: length = {expected_slope:.5f} × voltage")
print(f"Measured: length = {measured_slope:.5f} × voltage + {measured_intercept:.5f}")
print(f"Slope accuracy: {100 - slope_error:.1f}%")
print(f"R² = {r2:.6f}")
print(f"RMSE = {rmse:.6f}V")

In [ ]:
# FIGURE 3: Clean data presentation with dual axes and control voltage subfigure
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.3)

# Main plot: Length data with dual axes
ax1 = fig.add_subplot(gs[0, 0])

# Original data (faded)
color = 'lightblue'
ax1.plot(df['elapsed_s'], df['ch1_MEAN_V'], color=color, linewidth=1, 
         alpha=0.3, label='Raw Length Data')

# Filtered data (prominent)
color = 'tab:blue'
ax1.set_ylabel('Length Voltage (V)', color=color, fontsize=14)
ax1.plot(df['elapsed_s'], df['length_filtered'], color=color, linewidth=2.5, 
         alpha=0.9, label='Filtered Length Data')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Secondary Y-axis: Length in mm
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Length (mm)', color=color, fontsize=14)
ax2.plot(df['elapsed_s'], df['length_mm'], color=color, linewidth=2.5, 
         alpha=0.9, label='Length (mm)')
ax2.tick_params(axis='y', labelcolor=color)

# Mark cycle start
ax1.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)

ax1.set_title('Length Measurements (No Drift Correction Needed)', fontsize=16, fontweight='bold')

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Subfigure: Control voltage pattern
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(df['elapsed_s'], df['control_voltage'], color='red', linewidth=3, 
         alpha=0.8, label='Control Square Wave')
ax3.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)
ax3.set_xlabel('Time (seconds)', fontsize=14)
ax3.set_ylabel('Control (V)', fontsize=12)
ax3.set_title('Control Voltage Pattern (+0.6V → -0.8V, 300s each)', fontsize=14)
ax3.grid(True, alpha=0.3)
ax3.set_ylim(-1.2, 1.0)

# Add cycle annotations
if cycle_start_time > 0:
    cycle_centers = np.arange(cycle_start_time + 150, df['elapsed_s'].max(), 300)  # Every 300s
    for i, center in enumerate(cycle_centers[:8]):  # First 8 cycles
        if i % 2 == 0:
            ax3.annotate('+0.6V', xy=(center, 0.6), xytext=(center, 0.8),
                        ha='center', fontsize=10, alpha=0.7)
        else:
            ax3.annotate('-0.8V', xy=(center, -0.8), xytext=(center, -1.1),
                        ha='center', fontsize=10, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("EXPERIMENT SUMMARY - VERSION 2 (NO DRIFT CORRECTION)")
print("=" * 60)
print(f"📊 Total duration: {df['elapsed_s'].max()/60:.1f} minutes")
print(f"⏱️  Cycle start: {cycle_start_time:.1f} seconds")
print(f"🔧 Median filter size: {filter_size} points")

print(f"\n📏 LENGTH CONVERSION:")
print(f"   Formula: length (mm) = {length_conversion:.5f} × voltage (V)")
print(f"   Range: {df['length_mm'].min():.2f} to {df['length_mm'].max():.2f} mm")
print(f"   Total span: {df['length_mm'].max() - df['length_mm'].min():.2f} mm")

print(f"\n🎛️  CONTROL PATTERN (V2):")
print(f"   Voltage levels: +0.6V and -0.8V")
print(f"   Pattern: +0.6V (first 300s) → -0.8V (next 300s) → repeat")
print(f"   Cycle period: 600 seconds (300s each level)")
control_cycles = (df['elapsed_s'].max() - cycle_start_time) / 600
print(f"   Complete cycles: {control_cycles:.1f}")

print(f"\n📈 LINEAR RELATIONSHIP:")
print(f"   Expected slope: {expected_slope:.5f}")
print(f"   Measured slope: {measured_slope:.5f}")
print(f"   Accuracy: {100 - slope_error:.1f}%")
print(f"   R²: {r2:.6f}")
print(f"   No drift correction applied - data appears stable")

# Noise reduction from filtering only
original_noise = df['ch1_MEAN_V'].std()
filtered_noise = df['length_filtered'].std()
noise_reduction = (original_noise - filtered_noise) / original_noise * 100
print(f"\n🔊 NOISE REDUCTION:")
print(f"   Median filter: {noise_reduction:.1f}% noise reduction")
print(f"   Original std: {original_noise:.6f}V")
print(f"   Filtered std: {filtered_noise:.6f}V")